In [5]:
import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict
from groq import Groq

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


  # Define the structure of each action item
class ActionItem(BaseModel):
    model_config = ConfigDict(extra="forbid")

    task: str
    owner: str


# Define the overall output structure
class Summary(BaseModel):
    model_config = ConfigDict(extra="forbid")

    task_owner_list: list[ActionItem]


# Read the meeting transcript
with open("meeting.txt", "r", encoding="utf-8") as f:
    document = f.read()


prompt = f"""
Read the following meeting transcript and extract all action items.

For every action item, identify:
- task: What needs to be done, including the deadline if specified.
- owner: The person responsible for completing the task.

If the owner is not explicitly identified, use "Unassigned".
Do not invent task assignments.

Meeting transcript:
{document}
"""


try:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "action_items_summary",
                "strict": True,
                "schema": Summary.model_json_schema()
            }
        }
    )

    # Validate and parse the structured response
    action_items = Summary.model_validate_json(
        response.choices[0].message.content
    )

    # Display the structured output
    print(action_items.model_dump_json(indent=2))

    # Save the output to a JSON file
    with open("action_items.json", "w", encoding="utf-8") as f:
        json.dump(
            action_items.model_dump(),
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\nAction items saved to action_items.json")

except Exception as e:
    print(f"Something went wrong: {e}")

{
  "task_owner_list": [
    {
      "task": "Provide a defect update by 31 October 2026",
      "owner": "Amit Verma"
    },
    {
      "task": "Add test cases for invalid account statement date ranges",
      "owner": "Priya Sharma"
    },
    {
      "task": "Coordinate OTP integration testing and investigate payment gateway instability by 1 November 2026",
      "owner": "Neha Kapoor"
    },
    {
      "task": "Evaluate push notification automation",
      "owner": "Vikram Singh"
    }
  ]
}

Action items saved to action_items.json
